In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%load_ext memory_profiler

In [ ]:
from hydra import initialize, compose
from src.unit_proccessing import  *
from src.utils.chart_utils import *
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
with initialize(config_path='../configuration', version_base='1.1'):
    config = compose(config_name='main.yaml')

In [ ]:
features_class = UnitDataProcessing(config)
self = features_class

In [ ]:
df_item = features_class.df_item
df_unit = features_class.df_unit
df_unit_score = features_class.df_unit_score

In [ ]:
features_class.make_global_score()

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler,PowerTransformer, Normalizer

scaler = StandardScaler()
df = df_unit_score[self._score_columns]  # .astype(float).apply(adjustable_winsorize)
df = pd.DataFrame(scaler.fit_transform(df), columns=self._score_columns)
df['survey_version'] = df_unit_score['survey_version']
df['survey_label'] = df['survey_version'].apply(lambda x: False if int(x.split('_')[2])<13 else True)


df_melted = df[['survey_label']+self._score_columns].melt(id_vars='survey_label')
df_melted['survey_label'] = df_melted['survey_label'].replace({True:'Bad', False:'Good'})
df_melted = df_melted[~pd.isnull(df_melted['value'])]
df_melted['value'] == df_melted['value'].astype(float)
plt.figure(figsize=(12, 8))
sns.boxplot(x='variable', y='value', hue='survey_label', data=df_melted)
plt.title('Box plots of numeric columns by label')
plt.xticks(rotation=90)
plt.show()

In [ ]:
sns.boxplot(df_unit, x='survey_version', y='unit_risk_score')
plt.xticks(rotation=90)
plt.show()

In [ ]:
features_class._score_columns

In [ ]:

columns = [
    #'s__answer_changed',
 #'s__answer_duration_lower',
 #'s__answer_duration_upper',
 #'s__answer_hour_set',
 #'s__answer_selected_lower',
 #'s__answer_selected_upper',
 #'s__number_answered',
 #'s__pause_count',
 #'s__pause_duration',
 #'s__sequence_jump',
 #'s__time_changed',
 #'s__total_duration',
 #'s__total_elapse_lower',
 #'s__total_elapse_upper'
           ]
features_class.make_global_score(combine_resp_score=True, restricted_columns=columns)
df = features_class.df_unit.copy()
df['survey_label'] = df['survey_version'].apply(lambda x: False if int(x.split('_')[2])<13 else True)
make_top_perc_chart(df, target_label='survey_label', plot_first_percentiles=True, plot_perc_overall=True)

In [ ]:
df_unit['survey_label'] = df_unit['survey_version'].apply(lambda x: False if int(x.split('_')[2])<13 else True)
df_unit['color'] = df_unit['survey_label']>0
#df_unit['color'] = df_unit['survey_version'].apply(lambda x: 'Good' if int(x.split('_')[2])<13 else 'BBAD')
for col in features_class._score_columns:
    sns.scatterplot(y='unit_risk_score', x=col, data=df_unit, hue='color')
    plt.title(f"unit_risk_score vs. {col}")
    plt.show()



In [ ]:
sns.scatterplot(y='unit_risk_score', x='f__number_answered', data=df_unit[df_unit['f__number_answered']<150], hue='color')
plt.title(f"unit_risk_score vs. {col}")
plt.show()

In [ ]:
feature_name ='f__number_answered'
df[feature_name].value_counts()